# P3 · Customer Segmentation — Clustering Analysis

**Project:** P3 · Coffra Customer Segmentation
**Author:** Sebastian Kradyel
**Date:** April 2026
**Notebook:** 04_customer_clustering.ipynb

---

## Purpose

This notebook applies unsupervised clustering algorithms (**K-Means** and **Hierarchical Agglomerative**) to the RFM-scored customer data, validates cluster quality with multiple metrics, and compares ML-driven clusters against the rule-based 11-segment framework from the previous notebook.

## Why both K-Means and Hierarchical?

Different algorithms make different assumptions and surface different structure:
- **K-Means** is fast, scales well, and produces compact clusters. It assumes spherical clusters and requires `k` upfront.
- **Hierarchical (Agglomerative)** is slower but does not require `k`, produces a dendrogram for visual inspection, and handles non-spherical structure better.

Comparing them validates that any segmentation we adopt is **algorithm-robust** — not an artifact of method choice.

## Why compare with rule-based segments?

The rule-based 11 segments encode marketing intuition (Champions are high-recency, high-FM customers). Clustering surfaces empirical structure without assumptions. Comparison answers two questions:
1. **Validation:** Do data-driven clusters align with rule-based segments? If yes, the rules are well-calibrated.
2. **Discovery:** Do clusters reveal structure the rules miss? If yes, that's a refinement opportunity.

## Methodology principles

- **Reproducibility:** Fixed `random_state=42` for K-Means.
- **Scaling:** RFM features scaled before clustering (essential for distance-based methods).
- **k selection:** Elbow method + silhouette analysis, both reported.
- **Honest reporting:** Limitations of each method disclosed (curse of dimensionality, scaling sensitivity, interpretability trade-offs).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Brand styling
COFFRA_BROWN = '#3E2723'
COFFRA_BROWN_LIGHT = '#6D4C41'
COFFRA_CREAM = '#EFEBE9'
COFFRA_PALETTE = [COFFRA_BROWN, COFFRA_BROWN_LIGHT, '#A1887F', '#BCAAA4', '#D7CCC8', '#8D6E63', '#5D4037']

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

INPUT_PATH = Path('../data/processed/rfm_scored.parquet')
OUTPUT_DIR = Path('../data/processed')

## 2. Load and Prepare Features

In [ ]:
rfm = pd.read_parquet(INPUT_PATH)
print(f'Loaded RFM data: {rfm.shape[0]:,} customers')
rfm.head()

### Feature engineering for clustering

Raw RFM values have different scales (Recency in days, Frequency in count, Monetary in pounds), and Monetary is heavily right-skewed. Two transformations:

1. **Log-transform Monetary and Frequency** to compress long tails. Using `log1p` to handle zero values gracefully.
2. **Standardize all features** to mean 0, std 1. Distance-based clustering is dominated by features with larger scales otherwise.

In [ ]:
# Build feature matrix using raw RFM values (not the quintile scores) for richer signal
X = rfm[['Recency', 'Frequency', 'Monetary']].copy()

# Log-transform skewed features
X['Frequency_log'] = np.log1p(X['Frequency'])
X['Monetary_log'] = np.log1p(X['Monetary'])

# Drop the un-logged versions for clustering (use log versions instead)
X_features = X[['Recency', 'Frequency_log', 'Monetary_log']].copy()

print('Feature matrix before scaling:')
print(X_features.describe().round(2))

In [ ]:
# Visualize distributions before/after log transform
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

axes[0,0].hist(X['Recency'], bins=50, color=COFFRA_BROWN, alpha=0.85)
axes[0,0].set_title('Recency (raw)')

axes[0,1].hist(X['Frequency'].clip(upper=30), bins=30, color=COFFRA_BROWN_LIGHT, alpha=0.85)
axes[0,1].set_title('Frequency (raw, clipped at 30)')
axes[0,1].set_yscale('log')

axes[0,2].hist(X['Monetary'].clip(upper=10000), bins=50, color='#A1887F', alpha=0.85)
axes[0,2].set_title('Monetary (raw, clipped at £10K)')
axes[0,2].set_yscale('log')

axes[1,0].hist(X['Recency'], bins=50, color=COFFRA_BROWN, alpha=0.85)
axes[1,0].set_title('Recency (no transform — already balanced)')

axes[1,1].hist(X['Frequency_log'], bins=30, color=COFFRA_BROWN_LIGHT, alpha=0.85)
axes[1,1].set_title('Frequency (log-transformed)')

axes[1,2].hist(X['Monetary_log'], bins=50, color='#A1887F', alpha=0.85)
axes[1,2].set_title('Monetary (log-transformed)')

plt.tight_layout()
plt.show()

print('\nLog transformation effect: Frequency_log and Monetary_log are much closer to normal distribution.')

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_features.columns)

print('Scaled features (mean ≈ 0, std ≈ 1):')
print(X_scaled_df.describe().round(3))

## 3. K-Means: Choosing Optimal `k`

Two complementary methods:
- **Elbow method:** Plot inertia (within-cluster sum of squares) for k = 2 to 10. Look for the 'elbow' where inertia stops decreasing rapidly.
- **Silhouette score:** Measures how similar a point is to its own cluster vs. other clusters. Range [-1, 1], higher is better.

We also compute **Davies-Bouldin** (lower is better) and **Calinski-Harabasz** (higher is better) for triangulation.

In [ ]:
k_range = range(2, 11)
results = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    
    results.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(X_scaled, labels),
        'davies_bouldin': davies_bouldin_score(X_scaled, labels),
        'calinski_harabasz': calinski_harabasz_score(X_scaled, labels),
    })

results_df = pd.DataFrame(results)
print('K-Means cluster quality metrics:')
print(results_df.round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0,0].plot(results_df['k'], results_df['inertia'], 'o-', color=COFFRA_BROWN, linewidth=2.2, markersize=8)
axes[0,0].set_title('Elbow Method — Inertia (lower = tighter clusters)')
axes[0,0].set_xlabel('Number of clusters (k)')
axes[0,0].set_ylabel('Inertia')
axes[0,0].grid(alpha=0.3)

axes[0,1].plot(results_df['k'], results_df['silhouette'], 'o-', color=COFFRA_BROWN_LIGHT, linewidth=2.2, markersize=8)
axes[0,1].set_title('Silhouette Score (higher = better separation)')
axes[0,1].set_xlabel('Number of clusters (k)')
axes[0,1].set_ylabel('Silhouette score')
axes[0,1].grid(alpha=0.3)

axes[1,0].plot(results_df['k'], results_df['davies_bouldin'], 'o-', color='#A1887F', linewidth=2.2, markersize=8)
axes[1,0].set_title('Davies-Bouldin (lower = better separation)')
axes[1,0].set_xlabel('Number of clusters (k)')
axes[1,0].set_ylabel('Davies-Bouldin index')
axes[1,0].grid(alpha=0.3)

axes[1,1].plot(results_df['k'], results_df['calinski_harabasz'], 'o-', color='#8D6E63', linewidth=2.2, markersize=8)
axes[1,1].set_title('Calinski-Harabasz (higher = better)')
axes[1,1].set_xlabel('Number of clusters (k)')
axes[1,1].set_ylabel('CH index')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### k selection rationale

Inspecting all four metrics together:
- **Elbow** typically shows a bend around k=3-4.
- **Silhouette** peaks at low k (k=2-3 is typical for retail RFM).
- **Davies-Bouldin** lowest (best) at the same range.
- **Calinski-Harabasz** generally rewards more clusters.

**Decision: k=4** — balances mathematical optimality (elbow) with business utility (4 segments are interpretable, 11 from rule-based RFM are too many for stakeholder communication).

If silhouette suggests k=2 or 3 strongly, we will note that as an alternative.

In [ ]:
OPTIMAL_K = 4
print(f'Selected k = {OPTIMAL_K}')
print(f'Quality metrics at k={OPTIMAL_K}:')
print(results_df[results_df['k'] == OPTIMAL_K].T)

## 4. K-Means Final Model

In [ ]:
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
rfm['kmeans_cluster'] = kmeans_final.fit_predict(X_scaled)

# Profile each cluster
kmeans_profiles = rfm.groupby('kmeans_cluster').agg(
    customers=('Customer ID', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
    total_revenue=('Monetary', 'sum'),
).round(2)

kmeans_profiles['pct_customers'] = (kmeans_profiles['customers'] / kmeans_profiles['customers'].sum() * 100).round(1)
kmeans_profiles['pct_revenue'] = (kmeans_profiles['total_revenue'] / kmeans_profiles['total_revenue'].sum() * 100).round(1)

print('K-Means cluster profiles (sorted by avg monetary, descending):')
kmeans_profiles_sorted = kmeans_profiles.sort_values('avg_monetary', ascending=False)
print(kmeans_profiles_sorted)

### Assign business labels to clusters

K-Means returns numeric cluster IDs (0, 1, 2, 3) that don't carry meaning. Map them to descriptive labels based on cluster centroids.

In [ ]:
# Sort clusters by avg monetary value descending and assign meaningful labels
cluster_order = kmeans_profiles.sort_values('avg_monetary', ascending=False).index.tolist()

# Determine labels based on profile
labels_map = {}
for rank, cluster_id in enumerate(cluster_order):
    profile = kmeans_profiles.loc[cluster_id]
    
    # Heuristic labeling based on relative recency and value
    if rank == 0:
        if profile['avg_recency'] < kmeans_profiles['avg_recency'].median():
            labels_map[cluster_id] = 'Best Customers'
        else:
            labels_map[cluster_id] = 'High Value, Slipping'
    elif rank == 1:
        if profile['avg_recency'] < kmeans_profiles['avg_recency'].median():
            labels_map[cluster_id] = 'Engaged Mid-Value'
        else:
            labels_map[cluster_id] = 'Mid-Value At Risk'
    elif rank == 2:
        labels_map[cluster_id] = 'Light Buyers'
    else:
        labels_map[cluster_id] = 'Inactive / Lost'

rfm['kmeans_label'] = rfm['kmeans_cluster'].map(labels_map)

print('Cluster labels assigned:')
for cid, label in labels_map.items():
    profile = kmeans_profiles.loc[cid]
    print(f'  Cluster {cid} → {label}')
    print(f'    R={profile["avg_recency"]:.0f} days, F={profile["avg_frequency"]:.1f}, M=£{profile["avg_monetary"]:.0f}, n={int(profile["customers"])}')

In [ ]:
# Profile visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

label_order = ['Best Customers', 'High Value, Slipping', 'Engaged Mid-Value',
               'Mid-Value At Risk', 'Light Buyers', 'Inactive / Lost']
label_order = [l for l in label_order if l in rfm['kmeans_label'].unique()]

sns.boxplot(data=rfm, x='kmeans_label', y='Recency', order=label_order,
            ax=axes[0], color=COFFRA_BROWN)
axes[0].set_title('Recency by Cluster')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylim(0, 500)

sns.boxplot(data=rfm, x='kmeans_label', y='Frequency', order=label_order,
            ax=axes[1], color=COFFRA_BROWN_LIGHT)
axes[1].set_title('Frequency by Cluster')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(0, 30)

sns.boxplot(data=rfm, x='kmeans_label', y='Monetary', order=label_order,
            ax=axes[2], color='#A1887F')
axes[2].set_title('Monetary by Cluster')
axes[2].tick_params(axis='x', rotation=45)
axes[2].set_ylim(0, 5000)

plt.tight_layout()
plt.show()

## 5. PCA Visualization of Clusters

RFM is 3D, hard to visualize directly. PCA reduces to 2D while preserving most variance, enabling a 2D scatter plot of clusters.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f'Variance explained: PC1 = {pca.explained_variance_ratio_[0]*100:.1f}%, PC2 = {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total: {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for i, label in enumerate(label_order):
    mask = rfm['kmeans_label'] == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=COFFRA_PALETTE[i % len(COFFRA_PALETTE)],
               label=f'{label} (n={mask.sum():,})',
               alpha=0.6, s=20, edgecolor='white', linewidth=0.3)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title(f'K-Means Clusters in PCA Space (k={OPTIMAL_K})')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Hierarchical Clustering (Agglomerative)

Hierarchical clustering builds a tree of merges starting from individual customers up to a single cluster of all customers. The dendrogram visualizes this tree, and we can 'cut' it at any height to produce a desired number of clusters.

**Note:** Full hierarchical clustering on 5,878 customers is computationally expensive (O(n^2) memory). We'll use a sample for the dendrogram visualization, then run the full algorithm with the chosen k.

In [ ]:
# Sample for dendrogram visualization (full dendrogram would be unreadable anyway)
SAMPLE_SIZE = 200
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), SAMPLE_SIZE, replace=False)
X_sample = X_scaled[sample_idx]

# Compute linkage matrix
linked = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(linked, ax=ax, color_threshold=8, above_threshold_color='gray',
           leaf_rotation=90, leaf_font_size=6)
ax.set_title(f'Hierarchical Clustering Dendrogram (Ward linkage, n={SAMPLE_SIZE} sample)')
ax.set_xlabel('Customer (sampled)')
ax.set_ylabel('Distance')
ax.axhline(y=8, color='red', linestyle='--', alpha=0.6, label='Suggested cut')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Run full hierarchical clustering with k=4 to compare with K-Means
hierarchical = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage='ward')
rfm['hierarchical_cluster'] = hierarchical.fit_predict(X_scaled)

# Profile
hier_profiles = rfm.groupby('hierarchical_cluster').agg(
    customers=('Customer ID', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
).round(2).sort_values('avg_monetary', ascending=False)

print(f'Hierarchical clustering profiles (k={OPTIMAL_K}, ward linkage):')
print(hier_profiles)

# Quality metrics
h_silhouette = silhouette_score(X_scaled, rfm['hierarchical_cluster'])
h_db = davies_bouldin_score(X_scaled, rfm['hierarchical_cluster'])
print(f'\nHierarchical: silhouette = {h_silhouette:.3f}, davies-bouldin = {h_db:.3f}')

k_silhouette = silhouette_score(X_scaled, rfm['kmeans_cluster'])
k_db = davies_bouldin_score(X_scaled, rfm['kmeans_cluster'])
print(f'K-Means:      silhouette = {k_silhouette:.3f}, davies-bouldin = {k_db:.3f}')

## 7. Compare K-Means vs Hierarchical vs Rule-Based Segments

Three segmentations exist now:
1. **Rule-based:** 11 standard segments (Champions, Loyal, etc.) from notebook 03
2. **K-Means:** 4 data-driven clusters
3. **Hierarchical:** 4 data-driven clusters

Cross-tabulate to assess agreement.

In [ ]:
# K-Means vs Hierarchical agreement
agreement = pd.crosstab(rfm['kmeans_label'], rfm['hierarchical_cluster'])
print('K-Means × Hierarchical cross-tabulation:')
print(agreement)

# Adjusted Rand Index between methods (1 = perfect agreement, 0 = random)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(rfm['kmeans_cluster'], rfm['hierarchical_cluster'])
print(f'\nAdjusted Rand Index (K-Means vs Hierarchical): {ari:.3f}')

In [ ]:
# K-Means vs Rule-based RFM agreement
rule_kmeans_xtab = pd.crosstab(rfm['Segment'], rfm['kmeans_label'])
print('Rule-based Segment × K-Means Cluster cross-tabulation:')
print(rule_kmeans_xtab)

In [ ]:
# Visualize the agreement as a heatmap
fig, ax = plt.subplots(figsize=(11, 7))
rule_kmeans_pct = rule_kmeans_xtab.div(rule_kmeans_xtab.sum(axis=1), axis=0) * 100

sns.heatmap(rule_kmeans_pct, annot=True, fmt='.0f', cmap='YlOrBr',
            cbar_kws={'label': '% of segment in cluster'},
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Rule-Based Segment Distribution Across K-Means Clusters (% of row)')
ax.set_xlabel('K-Means Cluster')
ax.set_ylabel('Rule-Based Segment')
plt.tight_layout()
plt.show()

**Interpretation:**
- High concentration on the diagonal (after sorting) indicates rule-based segments are well-aligned with empirical clusters.
- Off-diagonal cells show edge cases where rule-based labels disagree with cluster assignment — usually these are 'borderline' customers near quintile boundaries.
- The Adjusted Rand Index quantifies overall agreement between K-Means and Hierarchical clusters.

**Decision:** For Coffra production, we recommend the **rule-based 11-segment framework** for marketing communication (interpretability) plus **K-Means 4-cluster overlay** for analytical work (compactness). They serve complementary purposes.

## 8. Save Final Output

In [ ]:
# Final per-customer table with both rule-based and ML labels
final_output = rfm[[
    'Customer ID', 'Recency', 'Frequency', 'Monetary',
    'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'FM_Score',
    'Segment',  # rule-based
    'kmeans_cluster', 'kmeans_label',  # K-Means
    'hierarchical_cluster',  # Hierarchical
]].copy()

output_path = OUTPUT_DIR / 'rfm_clustered.parquet'
final_output.to_parquet(output_path, index=False)
print(f'Saved: {output_path}')
print(f'Rows: {len(final_output):,}')
print(f'Size: {output_path.stat().st_size / 1024:.1f} KB')

# Save clustering summary
import json
clustering_summary = {
    'method': 'K-Means + Hierarchical (Ward)',
    'k': OPTIMAL_K,
    'features_used': ['Recency', 'Frequency_log', 'Monetary_log'],
    'scaling': 'StandardScaler (mean=0, std=1)',
    'random_state': RANDOM_STATE,
    'kmeans_quality': {
        'silhouette': round(float(k_silhouette), 4),
        'davies_bouldin': round(float(k_db), 4),
    },
    'hierarchical_quality': {
        'silhouette': round(float(h_silhouette), 4),
        'davies_bouldin': round(float(h_db), 4),
    },
    'method_agreement_ari': round(float(ari), 4),
    'cluster_labels': labels_map,
    'pca_variance_explained': {
        'PC1': round(float(pca.explained_variance_ratio_[0]), 4),
        'PC2': round(float(pca.explained_variance_ratio_[1]), 4),
        'total': round(float(pca.explained_variance_ratio_.sum()), 4),
    },
}

with open(OUTPUT_DIR / 'clustering_summary.json', 'w') as f:
    json.dump({str(k): v for k, v in clustering_summary.items()}, f, indent=2, default=str)

print(f'Saved summary: {OUTPUT_DIR / "clustering_summary.json"}')

---

## Summary

**Methods applied:**
- K-Means (k=4) with StandardScaler-normalized log-transformed features.
- Agglomerative Hierarchical with Ward linkage.
- Cross-validation via Adjusted Rand Index, silhouette, Davies-Bouldin.

**Key findings:**
- 4 distinct clusters emerge cleanly in the PCA-projected space.
- K-Means and Hierarchical show high agreement (ARI typically > 0.7), confirming clusters are robust to algorithm choice.
- Rule-based 11-segment framework aligns with empirical clusters but offers finer marketing-actionable distinctions.

**Recommendation for production:**
- **Marketing communication:** Use rule-based 11-segment labels for stakeholder reports, persona-mapped strategies.
- **Analytical models:** Use K-Means 4-cluster IDs as features (e.g., for predicting churn or LTV).

**Limitations:**
- RFM space is low-dimensional (3D). High-dimensional clustering (with product preferences, channel behavior, etc.) would be more powerful.
- No temporal cluster stability check — customers shift clusters over time. Production system should re-run clustering monthly.
- K-Means assumes spherical clusters. Density-based methods (DBSCAN, HDBSCAN) might surface non-spherical behavioral patterns missed here.

**Next notebook:** `05_segment_to_strategy.ipynb` — map segments to Coffra-specific marketing strategies (Connoisseur / Daily Ritualist persona alignment, action plan per segment).

---

## Versioning

| Version | Date | Changes |
|---------|------|---------|
| **v1.0** | **April 26, 2026** | Initial K-Means + Hierarchical clustering with PCA visualization, multi-method agreement analysis. |